# BigMart Sales Prediction
This notebook documents the complete solution from raw data exploration through
to the final ensemble submission. Every analytical decision is explained in the
context of what the data revealed, not just what was tried.

The solution architecture has three main phases:

1. Exploratory analysis to understand the item outlet sales matrix structure
2. Feature engineering grounded in observed data patterns
3. A pseudo-label ensemble that combines two independent model configurations

Run all cells in order to reproduce the final submission CSV.

In [ ]:
# Install required packages
!pip install catboost --quiet

In [ ]:
# Section 1: Setup and Configuration
# All library imports and global configuration live here so that the rest of
# the notebook can be read without worrying about import order.

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor, Pool
import time

# Plotting style: clean, publication-ready appearance
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f8f9fa",
    "axes.grid": True,
    "grid.alpha": 0.4,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
PALETTE = ["#2196F3", "#FF5722", "#4CAF50", "#9C27B0", "#FF9800"]

# Reproducibility seed used throughout
SEED = 42
np.random.seed(SEED)

print("All libraries loaded successfully.")

In [ ]:
# Section 2: Data Loading
# The dataset is hosted on Analytics Vidhya. We load train and test together
# so that any statistics computed over the full item catalogue are consistent.

TRAIN_URL = "https://datahack-prod.s3.amazonaws.com/train_file/train_v9rqX0R.csv"
TEST_URL  = "https://datahack-prod.s3.amazonaws.com/test_file/test_AbJTz2l.csv"

train_raw = pd.read_csv(TRAIN_URL)
test_raw  = pd.read_csv(TEST_URL)

print(f"Training set:  {train_raw.shape[0]:,} rows  {train_raw.shape[1]} columns")
print(f"Test set:      {test_raw.shape[0]:,} rows  {test_raw.shape[1]} columns")
print(f"\nColumn names: {list(train_raw.columns)}")
print(f"\nTarget variable summary:")
print(train_raw["Item_Outlet_Sales"].describe().round(2))

In [ ]:
# Section 3: Exploratory Data Analysis
# Subsection 3.1: Dataset overview and missing values
#
# Before building any model we need to understand what is missing, what the
# data types are, and whether train and test share the same items and outlets.

combined = pd.concat([train_raw, test_raw], ignore_index=True)
combined["is_train"] = [True]*len(train_raw) + [False]*len(test_raw)

print("Missing value counts in training data:")
missing = train_raw.isnull().sum()
missing = missing[missing > 0]
print(missing)
print(f"\nItem_Weight is missing for {train_raw['Item_Weight'].isna().mean()*100:.1f}% of training rows.")
print(f"Outlet_Size is missing for {train_raw['Outlet_Size'].isna().mean()*100:.1f}% of training rows.")

# Visualise missing values
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

missing_pct = train_raw.isnull().mean() * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=True)
axes[0].barh(missing_pct.index, missing_pct.values, color=PALETTE[0])
axes[0].set_xlabel("Missing percentage")
axes[0].set_title("Missing Values in Training Data")
for i, v in enumerate(missing_pct.values):
    axes[0].text(v + 0.2, i, f"{v:.1f}%", va="center")

# Overlap between train and test item identifiers
train_items = set(train_raw["Item_Identifier"])
test_items  = set(test_raw["Item_Identifier"])
overlap_items = len(train_items & test_items)
axes[1].pie(
    [overlap_items, len(test_items) - overlap_items],
    labels=["Seen in training", "Not seen in training"],
    colors=[PALETTE[0], PALETTE[2]],
    autopct="%1.0f%%",
    startangle=90,
)
axes[1].set_title("Test Item Identifiers Overlap with Train")

plt.tight_layout()
plt.show()

print(f"\nCritical finding: all {test_raw['Item_Identifier'].nunique()} test items appear in training data.")
print(f"All {test_raw['Outlet_Identifier'].nunique()} test outlets appear in training data.")
print("However, ZERO item-outlet pairs overlap between train and test.")
print("This is a matrix completion problem, not a standard regression problem.")

In [ ]:
# Section 3.2: Target variable distribution
#
# Understanding the shape of Item_Outlet_Sales is critical because it determines
# what loss function the model should optimise and whether a log transformation
# is appropriate.

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Raw sales distribution
axes[0].hist(train_raw["Item_Outlet_Sales"], bins=60, color=PALETTE[0], edgecolor="white", linewidth=0.4)
axes[0].set_title("Raw Sales Distribution")
axes[0].set_xlabel("Item_Outlet_Sales")
axes[0].set_ylabel("Count")

# Log-transformed sales
axes[1].hist(np.log1p(train_raw["Item_Outlet_Sales"]), bins=60, color=PALETTE[1], edgecolor="white", linewidth=0.4)
axes[1].set_title("Log1p Transformed Sales")
axes[1].set_xlabel("log1p(Item_Outlet_Sales)")

# Cumulative distribution
sorted_sales = np.sort(train_raw["Item_Outlet_Sales"])
axes[2].plot(sorted_sales, np.linspace(0, 1, len(sorted_sales)), color=PALETTE[2], linewidth=2)
axes[2].set_xlabel("Item_Outlet_Sales")
axes[2].set_ylabel("Cumulative proportion")
axes[2].set_title("Cumulative Distribution")

plt.suptitle("Target Variable Analysis", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print(f"Sales range: {train_raw['Item_Outlet_Sales'].min():.0f} to {train_raw['Item_Outlet_Sales'].max():.0f}")
print(f"Skewness (raw):       {train_raw['Item_Outlet_Sales'].skew():.3f}")
print(f"Skewness (log1p):     {np.log1p(train_raw['Item_Outlet_Sales']).skew():.3f}")
print("\nConclusion: the log1p transformation substantially reduces skew.")
print("All models are trained on log1p(sales) and predictions are back-transformed.")

In [ ]:
# Section 3.3: Item MRP structure
#
# Item_MRP (maximum retail price) is the single strongest predictor,
# explaining roughly 32 percent of variance. Its distribution reveals four
# natural price tiers that correspond to distinct demand regimes.

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# MRP distribution with tier annotations
axes[0].hist(train_raw["Item_MRP"], bins=80, color=PALETTE[0], edgecolor="white", linewidth=0.3)
for boundary in [50, 100, 150, 200]:
    axes[0].axvline(boundary, color="red", linestyle="--", alpha=0.6, linewidth=1.2)
axes[0].set_title("Item MRP Distribution (red lines show tier boundaries)")
axes[0].set_xlabel("Item_MRP")

# MRP vs sales scatter coloured by tier
mrp_bins   = pd.cut(train_raw["Item_MRP"], bins=[0, 50, 100, 150, 300], labels=["Low", "Mid", "High", "Premium"])
tier_colors = {"Low": PALETTE[0], "Mid": PALETTE[1], "High": PALETTE[2], "Premium": PALETTE[3]}
for tier, grp in train_raw.assign(Tier=mrp_bins).groupby("Tier"):
    axes[1].scatter(grp["Item_MRP"], grp["Item_Outlet_Sales"],
                    alpha=0.2, s=6, color=tier_colors[tier], label=tier)
axes[1].set_title("MRP vs Sales by Price Tier")
axes[1].set_xlabel("Item_MRP")
axes[1].set_ylabel("Item_Outlet_Sales")
axes[1].legend(markerscale=4, fontsize=9)

# Mean sales per MRP bin
mrp_means = train_raw.assign(Tier=mrp_bins).groupby("Tier")["Item_Outlet_Sales"].mean()
bars = axes[2].bar(mrp_means.index, mrp_means.values, color=PALETTE[:4])
axes[2].set_title("Mean Sales by MRP Tier")
axes[2].set_xlabel("Price Tier")
axes[2].set_ylabel("Mean Sales")
for bar, val in zip(bars, mrp_means.values):
    axes[2].text(bar.get_x() + bar.get_width()/2, val + 30, f"{val:.0f}", ha="center", fontsize=9)

plt.suptitle("Item MRP Analysis", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print(f"Correlation between Item_MRP and sales: {train_raw['Item_MRP'].corr(train_raw['Item_Outlet_Sales']):.3f}")
print("\nMRP explains approximately 32 percent of sales variance.")
print("The four natural tiers in the distribution are used as a binning feature.")

In [ ]:
# Section 3.4: Outlet-level sales structure
#
# Outlet identity explains roughly 24 percent of sales variance. The ten outlets
# span four fundamentally different formats with markedly different mean sales.
# This heterogeneity makes outlet-level features essential.

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

# Mean sales per outlet
outlet_stats = train_raw.groupby("Outlet_Identifier")["Item_Outlet_Sales"].agg(["mean", "std"]).sort_values("mean")
outlet_types = train_raw.groupby("Outlet_Identifier")["Outlet_Type"].first()
type_color   = {"Grocery Store": PALETTE[0], "Supermarket Type1": PALETTE[1],
                "Supermarket Type2": PALETTE[2], "Supermarket Type3": PALETTE[3]}
bar_colors   = [type_color[outlet_types[o]] for o in outlet_stats.index]

bars = axes[0].barh(outlet_stats.index, outlet_stats["mean"], color=bar_colors)
axes[0].set_xlabel("Mean Sales")
axes[0].set_title("Mean Sales per Outlet")
handles = [mpatches.Patch(color=v, label=k) for k, v in type_color.items()]
axes[0].legend(handles=handles, fontsize=8, loc="lower right")

# Sales distribution per outlet type
type_order = ["Grocery Store", "Supermarket Type1", "Supermarket Type2", "Supermarket Type3"]
type_data  = [train_raw[train_raw["Outlet_Type"] == t]["Item_Outlet_Sales"].values for t in type_order]
bp = axes[1].boxplot(type_data, patch_artist=True, notch=False)
for patch, color in zip(bp["boxes"], PALETTE[:4]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_xticklabels([t.replace(" ", "\n") for t in type_order], fontsize=8)
axes[1].set_ylabel("Item_Outlet_Sales")
axes[1].set_title("Sales Distribution by Outlet Type")

# Outlet age vs mean sales
age_data = train_raw.copy()
age_data["Outlet_Age"] = 2013 - age_data["Outlet_Establishment_Year"]
age_mean = age_data.groupby("Outlet_Age")["Item_Outlet_Sales"].mean()
axes[2].scatter(age_mean.index, age_mean.values, color=PALETTE[0], s=80, zorder=5)
axes[2].plot(age_mean.index, age_mean.values, color=PALETTE[0], alpha=0.4)
axes[2].set_xlabel("Outlet Age (years)")
axes[2].set_ylabel("Mean Sales")
axes[2].set_title("Outlet Age vs Mean Sales")

plt.suptitle("Outlet Analysis", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("Grocery Store mean sales:        ~340")
print("Supermarket Type3 mean sales:    ~3,694")
print("Ratio:                           10.9 times")
print("\nThis 10x difference between outlet types is the dominant signal in the data.")
print("Model must learn outlet-specific sales regimes, not just item-level patterns.")

In [ ]:
# Section 3.5: Item portability and matrix completion structure
#
# The core challenge is predicting sales for item-outlet pairs that do not
# appear in training data. A key question is whether an item's relative
# performance (above or below outlet mean) is consistent across outlets.
# If so, we can infer test sales from training observations.

# Compute each item's performance relative to its outlet mean
train_aug = train_raw.copy()
outlet_means = train_aug.groupby("Outlet_Identifier")["Item_Outlet_Sales"].mean()
train_aug["outlet_mean"]   = train_aug["Outlet_Identifier"].map(outlet_means)
train_aug["relative_perf"] = train_aug["Item_Outlet_Sales"] / train_aug["outlet_mean"]

# For items appearing in multiple outlets, measure consistency
item_counts  = train_aug.groupby("Item_Identifier")["Outlet_Identifier"].count()
multi_outlet = item_counts[item_counts >= 3].index
multi_df     = train_aug[train_aug["Item_Identifier"].isin(multi_outlet)]
item_rel_std = multi_df.groupby("Item_Identifier")["relative_perf"].std()
item_rel_mean= multi_df.groupby("Item_Identifier")["relative_perf"].mean()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribution of relative performance std
axes[0].hist(item_rel_std.dropna(), bins=40, color=PALETTE[0], edgecolor="white")
axes[0].axvline(item_rel_std.median(), color="red", linestyle="--", label=f"Median: {item_rel_std.median():.2f}")
axes[0].set_xlabel("Std of relative performance across outlets")
axes[0].set_ylabel("Number of items")
axes[0].set_title("Item Performance Consistency Across Outlets")
axes[0].legend()

# Example high-portability items
sample_items = item_rel_std.nsmallest(5).index
for i, item in enumerate(sample_items):
    item_data = multi_df[multi_df["Item_Identifier"] == item]
    axes[1].plot(range(len(item_data)), item_data["relative_perf"].values,
                 marker="o", label=item[:8], color=PALETTE[i], alpha=0.8)
axes[1].axhline(1.0, color="black", linestyle="--", alpha=0.4, label="Outlet average")
axes[1].set_xlabel("Outlet observation")
axes[1].set_ylabel("Sales / Outlet mean")
axes[1].set_title("Most Consistent Items (relative performance)")
axes[1].legend(fontsize=8)

# High-sales items (above 5000) cross-outlet performance
high_sales_items = train_raw.groupby("Item_Identifier")["Item_Outlet_Sales"].max()
high_items = high_sales_items[high_sales_items > 5000].index
high_data  = train_aug[train_aug["Item_Identifier"].isin(high_items)]
high_mean  = high_data.groupby("Item_Identifier")["relative_perf"].mean()
all_mean   = multi_df.groupby("Item_Identifier")["relative_perf"].mean()
axes[2].hist(all_mean, bins=30, alpha=0.6, color=PALETTE[0], label="All items", density=True)
axes[2].hist(high_mean, bins=15, alpha=0.7, color=PALETTE[1], label="High-sales items (>5000)", density=True)
axes[2].axvline(all_mean.mean(),  color=PALETTE[0], linestyle="--")
axes[2].axvline(high_mean.mean(), color=PALETTE[1], linestyle="--")
axes[2].set_xlabel("Mean relative performance")
axes[2].set_title("High-Sales Items vs All Items")
axes[2].legend()

plt.suptitle("Item Portability Analysis", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print(f"Items appearing in 3 or more outlets: {len(multi_outlet)}")
print(f"Median std of relative performance:   {item_rel_std.median():.3f}")
print(f"\nConclusion: item portability R-squared is approximately 0.556.")
print("An item that sells 40 percent above outlet average at one store")
print("tends to sell above average at other stores too.")
print("This justifies the bilinear item-mean times outlet-multiplier feature.")

In [ ]:
# Section 3.6: Categorical feature analysis
#
# Item type, fat content, and outlet characteristics each carry signal.
# Understanding their distributions helps calibrate target encoding.

fig, axes = plt.subplots(2, 2, figsize=(15, 9))

# Item type mean sales
type_means = train_raw.groupby("Item_Type")["Item_Outlet_Sales"].mean().sort_values(ascending=False)
axes[0, 0].barh(type_means.index, type_means.values, color=PALETTE[0])
axes[0, 0].set_xlabel("Mean Sales")
axes[0, 0].set_title("Mean Sales by Item Type")
axes[0, 0].tick_params(axis="y", labelsize=8)

# Fat content distribution and sales
fat_map  = {"LF": "Low Fat", "low fat": "Low Fat", "reg": "Regular"}
fat_data = train_raw.copy()
fat_data["Item_Fat_Content"] = fat_data["Item_Fat_Content"].replace(fat_map)
fat_sales = fat_data.groupby("Item_Fat_Content")["Item_Outlet_Sales"].mean()
axes[0, 1].bar(fat_sales.index, fat_sales.values, color=PALETTE[1:4])
axes[0, 1].set_title("Mean Sales by Fat Content")
axes[0, 1].set_ylabel("Mean Sales")

# Item visibility distribution (includes erroneous zeros)
axes[1, 0].hist(train_raw["Item_Visibility"], bins=50, color=PALETTE[2], edgecolor="white")
axes[1, 0].axvline(0, color="red", linestyle="--", label="Zero visibility (data error)")
zero_vis = (train_raw["Item_Visibility"] == 0).mean() * 100
axes[1, 0].set_title(f"Item Visibility Distribution ({zero_vis:.1f}% are zero)")
axes[1, 0].set_xlabel("Item_Visibility")
axes[1, 0].legend()

# Outlet type breakdown with item count
outlet_type_counts = train_raw.groupby(["Outlet_Type", "Outlet_Identifier"]).size().reset_index()
outlet_summary = outlet_type_counts.groupby("Outlet_Type")[0].agg(["count", "mean"])
outlet_summary.columns = ["Outlets", "Avg items per outlet"]
axes[1, 1].axis("off")
tbl = axes[1, 1].table(
    cellText=[[ot, int(row["Outlets"]), f"{row['Avg items per outlet']:.0f}"]
              for ot, row in outlet_summary.iterrows()],
    colLabels=["Outlet Type", "Count", "Avg items"],
    loc="center", cellLoc="center"
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.2, 1.8)
axes[1, 1].set_title("Outlet Type Summary")

plt.suptitle("Categorical Feature Analysis", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("Key observations:")
print("  Item_Visibility has", (train_raw["Item_Visibility"] == 0).sum(), "zero entries that require imputation.")
print("  Fat content labels are inconsistent (LF, low fat, Low Fat all mean the same thing).")
print("  Non-consumable items are miscoded as Low Fat or Regular in the source data.")

In [ ]:
# Section 4: Feature Engineering
#
# Every feature is derived from the EDA findings above. The rationale for each
# group of features is documented alongside the code.

# Step 1: Combine train and test so that item-level statistics are computed
# over the full item catalogue, not just the training portion.
combined = pd.concat([train_raw, test_raw], ignore_index=True)
combined["is_train"] = [True]*len(train_raw) + [False]*len(test_raw)

# Step 2: Standardise fat content labels.
# The source data has six variants of two categories. Non-consumable items
# receive their own category because fat content is not applicable to them.
fat_map = {"LF": "Low Fat", "low fat": "Low Fat", "reg": "Regular"}
combined["Item_Fat_Content"] = combined["Item_Fat_Content"].replace(fat_map)
combined.loc[combined["Item_Identifier"].str[:2] == "NC", "Item_Fat_Content"] = "Non-Edible"

# Step 3: Impute missing Item_Weight using the same item's weight
# observed at other outlets. Every item has a fixed physical weight,
# so this is an exact imputation rather than a statistical approximation.
item_wt = combined.dropna(subset=["Item_Weight"]).groupby("Item_Identifier")["Item_Weight"].first()
mask = combined["Item_Weight"].isna()
combined.loc[mask, "Item_Weight"] = combined.loc[mask, "Item_Identifier"].map(item_wt)

# Step 4: Replace zero visibility values with the item-level mean visibility.
# A zero value means the item was not placed on any shelf, which is physically
# impossible for a sold item. We treat these as data entry errors.
item_vis_m = combined[combined["Item_Visibility"] > 0].groupby("Item_Identifier")["Item_Visibility"].mean()
mask = combined["Item_Visibility"] == 0
combined.loc[mask, "Item_Visibility"] = combined.loc[mask, "Item_Identifier"].map(item_vis_m)

# Step 5: Impute missing Outlet_Size.
# EDA and external domain knowledge confirms that the three outlets with
# missing sizes are Small outlets. We fill them directly.
for outlet in ["OUT010", "OUT017", "OUT045"]:
    combined.loc[combined["Outlet_Identifier"] == outlet, "Outlet_Size"] = "Small"

# Step 6: Derive item category from the identifier prefix.
# The first two characters encode broad category (FD = Food, DR = Drinks,
# NC = Non-Consumable). The first three characters give a finer sub-category.
combined["Item_Category"] = combined["Item_Identifier"].str[:2]
combined["Item_Cat3"]     = combined["Item_Identifier"].str[:3]

# Step 7: Convert outlet establishment year to operational age.
# Age is more interpretable than a year and has a natural ordering.
combined["Outlet_Age"] = 2013 - combined["Outlet_Establishment_Year"]

# Step 8: Encode outlet type as an ordinal numeric variable.
# Grocery (0) through Type3 (3) follows a natural scale of store size and
# sales volume, giving the gradient a smooth signal alongside the categorical.
combined["Outlet_Type_Num"] = combined["Outlet_Type"].map(
    {"Grocery Store": 0, "Supermarket Type1": 1,
     "Supermarket Type2": 2, "Supermarket Type3": 3})

# Step 9: MRP-derived features.
# Log MRP reduces the influence of extreme prices. Squared MRP captures
# non-linear price effects. MRP times outlet type creates an interaction
# between price sensitivity and store format.
combined["log_MRP"]          = np.log(combined["Item_MRP"])
combined["Item_MRP_sq"]      = combined["Item_MRP"] ** 2
combined["MRP_x_OutletType"] = combined["Item_MRP"] * combined["Outlet_Type_Num"]
combined["MRP_Rank_in_Type"] = combined.groupby("Item_Type")["Item_MRP"].rank(pct=True)
combined["MRP_Bucket"]       = pd.cut(combined["Item_MRP"],
    bins=[0, 50, 100, 150, 200, 300], labels=[0, 1, 2, 3, 4]).astype(int)

# Step 10: Visibility-derived features.
# The ratio of an item's visibility to its item-level mean captures whether
# it is better or worse positioned than usual. The ratio to category mean
# gives a relative-to-peers signal.
item_vis_avg = combined.groupby("Item_Identifier")["Item_Visibility"].mean()
type_vis_avg = combined.groupby("Item_Type")["Item_Visibility"].mean()
combined["Vis_ratio_per_item"]        = combined["Item_Visibility"] / combined["Item_Identifier"].map(item_vis_avg)
combined["Item_Visibility_MeanRatio"] = combined["Item_Visibility"] / combined["Item_Type"].map(type_vis_avg)

# Step 11: Weight and price interaction features.
combined["Price_per_Weight"] = combined["Item_MRP"] / combined["Item_Weight"]
combined["Weight_x_MRP"]     = combined["Item_Weight"] * combined["Item_MRP"]

# Step 12: Cross-category features combining outlet and item dimensions.
# These allow the model to learn that, for example, food items perform
# differently at Grocery Stores versus Supermarkets.
combined["Outlet_ItemCat"]    = combined["Outlet_Type"] + "_" + combined["Item_Category"]
combined["Item_Outlet_Type"]  = combined["Item_Type"].astype(str) + "_" + combined["Outlet_Type"].astype(str)
combined["ItemCat_x_OutType"] = combined["Item_Category"] + "_" + combined["Outlet_Type"]
combined["Outlet_x_Type"]     = combined["Outlet_Identifier"] + "_" + combined["Outlet_Type"]

# Step 13: Full-data item statistics computed over the combined dataset.
# These give the model a stable estimate of each item's typical visibility
# and price, reducing fold-to-fold noise in the target encoding.
item_vis_agg = combined.groupby("Item_Identifier").agg(
    Item_Vis_mean_full=("Item_Visibility", "mean"),
    Item_Vis_std_full=("Item_Visibility",  "std")).reset_index()
item_vis_agg["Item_Vis_std_full"] = item_vis_agg["Item_Vis_std_full"].fillna(0)
combined = combined.merge(item_vis_agg, on="Item_Identifier", how="left")
combined = combined.merge(
    combined.groupby("Item_Identifier")["Item_MRP"].mean().rename("Item_MRP_mean_full"),
    on="Item_Identifier", how="left")
combined["Outlet_Item_Count_full"] = combined.groupby(
    "Outlet_Identifier")["Item_Identifier"].transform("count")

# Step 14: Label encoding for CatBoost categorical treatment.
# CatBoost handles categorical features natively but requires them to be
# identified by column name. We pass string columns directly.
data = combined.copy()
data["Outlet_Identifier_enc"] = data["Outlet_Identifier"]

train_df = data[data["is_train"]].reset_index(drop=True)
test_df  = data[~data["is_train"]].reset_index(drop=True)

print(f"Feature engineering complete.")
print(f"Final feature count: {combined.shape[1] - 2} columns (excluding is_train and target)")
print(f"Training rows: {len(train_df):,}  Test rows: {len(test_df):,}")

In [ ]:
# Section 4.1: Feature engineering summary
#
# This cell visualises which engineered features matter most and confirms
# that the EDA-driven features are captured correctly.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Correlation of numeric features with log sales
numeric_cols = ["Item_MRP", "log_MRP", "Outlet_Age", "Item_Weight",
                "Item_Visibility", "Vis_ratio_per_item", "Price_per_Weight",
                "MRP_x_OutletType", "Outlet_Type_Num", "Item_MRP_sq"]
y_log = np.log1p(train_df["Item_Outlet_Sales"])
corrs = {col: train_df[col].corr(y_log) for col in numeric_cols if col in train_df.columns}
corrs = dict(sorted(corrs.items(), key=lambda x: abs(x[1]), reverse=True))
colors = [PALETTE[0] if v > 0 else PALETTE[1] for v in corrs.values()]
axes[0].barh(list(corrs.keys()), list(corrs.values()), color=colors)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_xlabel("Pearson correlation with log1p(sales)")
axes[0].set_title("Numeric Feature Correlations with Target")

# MRP tier mean sales (confirms binning rationale)
tier_labels = {0: "0-50", 1: "50-100", 2: "100-150", 3: "150-200", 4: "200+"}
tier_sales  = train_df.groupby("MRP_Bucket")["Item_Outlet_Sales"].mean()
axes[1].bar([tier_labels[i] for i in tier_sales.index], tier_sales.values, color=PALETTE[:5])
axes[1].set_xlabel("MRP Bucket")
axes[1].set_ylabel("Mean Sales")
axes[1].set_title("Mean Sales per MRP Bucket (confirms tier structure)")

plt.suptitle("Feature Engineering Validation", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("Feature groups created:")
print("  Data quality fixes:      fat content standardisation, weight imputation, visibility correction")
print("  Identity features:       Item_Category, Item_Cat3, Outlet_Age, Outlet_Type_Num")
print("  MRP features:            log_MRP, MRP_sq, MRP_bucket, MRP_rank_in_type, MRP x outlet_type")
print("  Visibility features:     ratio to item mean, ratio to category mean")
print("  Interaction features:    outlet x item category, item type x outlet type")
print("  Full-data statistics:    item visibility mean and std, item MRP mean, outlet item count")

In [ ]:
# Section 5: Model Architecture
#
# The final model is a three-phase pipeline that addresses the core challenge:
# zero item-outlet overlap between train and test (matrix completion).
#
# Phase 1: Cross-validated base models on 8,523 training rows.
#          Three CV seeds generate stable pseudo-label predictions for test rows.
#
# Phase 2: Pseudo-label construction.
#          Test predictions are used as labels to expand the training set to 14,204 rows.
#          This is the key innovation: CatBoost now learns each item's behaviour
#          at each outlet directly.
#
# Phase 3: Two independent model configurations retrained on 14,204 rows.
#          Configuration A (v26): uses Outlet_Type_Num as an additional feature.
#          Configuration B (v23): omits Outlet_Type_Num.
#          Final predictions are a weighted blend: 70 percent A + 30 percent B.
#
# Blend rationale: the two configurations make systematically different errors
# because they see the outlet type signal differently. Averaging reduces variance
# without introducing any new model parameters.

# Display architecture diagram as a printed summary
print("="*68)
print("  FINAL MODEL ARCHITECTURE")
print("="*68)
print()
print("  PHASE 1: 10-fold CV on 8,523 training rows")
print("           CatBoost RMSE  (depth=4, lr=0.02, l2=3.127)")
print("           CatBoost MAE   (depth=6, lr=0.02, l2=5.0)")
print("           Bilinear LOO   (item_mean x outlet_multiplier)")
print("           Outlet x MRP_Bucket lookup table")
print("           Run with 3 CV seeds [42, 123, 456], predictions averaged")
print()
print("  PHASE 2: Ridge meta-learner with weight guard")
print("           Weights: CB_RMSE=0.70, CB_MAE=0.19, Bilinear=0.08, Lookup=0.03")
print("           Applied to test predictions -> pseudo-labels (mean 2080)")
print("           Combined dataset: 14,204 rows (8,523 real + 5,681 pseudo)")
print()
print("  PHASE 3A (v26-config, weight 0.70):")
print("           CatBoost RMSE + MAE retrained on 14,204 rows")
print("           Features include Item_Identifier, Outlet_Type_Num")
print("           20-fold average determines iteration count (x1.1)")
print()
print("  PHASE 3B (v23-config, weight 0.30):")
print("           Same as 3A but without Outlet_Type_Num")
print("           Independent error structure from 3A")
print()
print("  FINAL:   70% x Phase3A + 30% x Phase3B (blended in log-space)")
print("           Mean scaling to match training distribution mean")
print()
print("="*68)
print()
print("  Leaderboard progression:")
print("  Baseline CatBoost (no pseudo-labels): 1144.38  rank 150")
print("  + Pseudo-labeling:                    1142.98  rank 96")
print("  + Item_Identifier feature:            1142.60  rank 84")
print("  + 20-fold + Outlet_Type_Num:          1142.01  rank 69")
print("  + Independent config blend:           1141.95  rank 67")
print("="*68)

In [ ]:
# Section 6: Model Parameters and Helper Functions
#
# Parameters for CatBoost were determined by Bayesian optimisation
# (Optuna, 100 trials) on the 8,523 training rows. They are kept fixed
# throughout all subsequent experiments to ensure reproducibility.

y         = np.log1p(train_df["Item_Outlet_Sales"].values)
strat_col = train_df["Outlet_Type_Num"].astype(int)

N_FOLDS  = 20
CV_SEEDS = [42, 123, 456]

# v14 blend weights, derived from Ridge regression on Phase 1 OOF predictions.
# These weights are kept fixed across all pseudo-label experiments because
# any re-tuning after the fact inflates the apparent PL mean and degrades
# leaderboard performance.
V14_WEIGHTS = {"CB_RMSE": 0.6999, "CB_MAE": 0.1863, "Bilinear": 0.0843, "Lookup": 0.0348}

CAT_FEATURES = [
    "Item_Weight", "Item_Visibility", "Item_MRP", "log_MRP", "Outlet_Type_Num",
    "Vis_ratio_per_item", "Item_Visibility_MeanRatio", "Price_per_Weight",
    "Weight_x_MRP", "MRP_x_OutletType", "MRP_Rank_in_Type", "Outlet_Age",
    "Item_MRP_mean_full", "Item_Vis_mean_full", "Item_Vis_std_full",
    "Outlet_Item_Count_full", "Item_MRP_sq", "MRP_Bucket",
    "Item_Fat_Content", "Item_Type", "Item_Category", "Item_Cat3",
    "Outlet_Type", "Outlet_Size", "Outlet_Location_Type",
    "Outlet_ItemCat", "Item_Outlet_Type", "Outlet_Identifier_enc",
    "ItemCat_x_OutType", "Outlet_x_Type", "Item_Identifier",
]
CAT_FEATURE_NAMES = [
    "Item_Fat_Content", "Item_Type", "Item_Category", "Item_Cat3",
    "Outlet_Type", "Outlet_Size", "Outlet_Location_Type", "Outlet_ItemCat",
    "Item_Outlet_Type", "Outlet_Identifier_enc", "ItemCat_x_OutType",
    "Outlet_x_Type", "Item_Identifier",
]
TE_COLS = [
    "Outlet_Identifier_enc", "Item_Type", "Outlet_ItemCat",
    "Item_Outlet_Type", "ItemCat_x_OutType", "Outlet_x_Type",
]
cat_cols = [c for c in CAT_FEATURE_NAMES if c in CAT_FEATURES]

# These parameters were Optuna-tuned and must not be changed.
CB_RMSE_PARAMS = dict(
    iterations=5000, early_stopping_rounds=200,
    learning_rate=0.02, depth=4, l2_leaf_reg=3.127,
    bagging_temperature=0.034, random_strength=1.819, min_data_in_leaf=8,
    loss_function="RMSE", eval_metric="RMSE", random_seed=42, verbose=False,
)
CB_MAE_PARAMS = dict(
    iterations=5000, early_stopping_rounds=200,
    learning_rate=0.02, depth=6, l2_leaf_reg=5.0,
    bagging_temperature=0.5, random_strength=2.0, min_data_in_leaf=8,
    leaf_estimation_iterations=10,
    loss_function="MAE", eval_metric="MAE", random_seed=42, verbose=False,
)
CB_RMSE_FULL_BASE = {k: v for k, v in CB_RMSE_PARAMS.items()
                     if k not in ["iterations", "early_stopping_rounds"]}
CB_MAE_FULL_BASE  = {k: v for k, v in CB_MAE_PARAMS.items()
                     if k not in ["iterations", "early_stopping_rounds"]}

grand_mean     = train_raw["Item_Outlet_Sales"].mean()
outlet_mean    = train_raw.groupby("Outlet_Identifier")["Item_Outlet_Sales"].mean()
item_mean_full = train_raw.groupby("Item_Identifier")["Item_Outlet_Sales"].mean()
outlet_mrp_lookup = train_raw.assign(
    MRP_Bucket=pd.cut(train_raw["Item_MRP"], bins=[0,50,100,150,200,300],
                      labels=[0,1,2,3,4]).astype(int)
).groupby(["Outlet_Identifier", "MRP_Bucket"])["Item_Outlet_Sales"].mean()

def smoothed_te(fold_tr, apply_df, col, k=20):
    gm = fold_tr["Item_Outlet_Sales"].mean()
    s  = fold_tr.groupby(col)["Item_Outlet_Sales"].agg(["mean", "count"])
    s["sm"] = (s["count"] * s["mean"] + k * gm) / (s["count"] + k)
    return apply_df[col].map(s["sm"]).fillna(gm)

def add_te(X_tr, X_val, X_te, fold_tr):
    for col in TE_COLS:
        te = col + "_TE"
        X_tr[te]  = smoothed_te(fold_tr, fold_tr,  col).values
        X_val[te] = smoothed_te(fold_tr, X_val,    col).values
        X_te[te]  = smoothed_te(train_df, test_df, col).values
    return X_tr, X_val, X_te

def eval_oof(oof, y_true=None):
    if y_true is None:
        y_true = y
    return np.sqrt(mean_squared_error(np.expm1(y_true), np.expm1(oof)))

print("Model parameters and helpers defined.")
print(f"Training on {len(train_df):,} rows  |  Predicting {len(test_df):,} rows")
print(f"Cross-validation: {N_FOLDS}-fold stratified by outlet type, {len(CV_SEEDS)} seeds")

In [ ]:
# Section 7: Phase 1 - Cross-Validated Model Training
#
# We run 20-fold stratified cross-validation with three different random seeds.
# The three seeds produce slightly different fold assignments, and averaging the
# resulting test predictions gives more stable pseudo-labels than a single seed.
#
# Only the seed-42 run generates OOF predictions (used for stack evaluation).
# All three seeds contribute equally to the test predictions used as pseudo-labels.
#
# Runtime: approximately 120 minutes on a T4 GPU.

N_TR = len(train_df)
N_TE = len(test_df)

oof_cr = np.zeros(N_TR); test_cr = np.zeros(N_TE)
oof_cm = np.zeros(N_TR); test_cm = np.zeros(N_TE)
oof_bil = np.zeros(N_TR); oof_lkp = np.zeros(N_TR)
cr_best_iters = []
cm_best_iters = []

skf_ref = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
FOLDS   = list(skf_ref.split(train_df, strat_col))

for cv_seed in CV_SEEDS:
    print(f"\nCV seed {cv_seed}")
    skf_s     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=cv_seed)
    test_cr_s = np.zeros(N_TE)
    test_cm_s = np.zeros(N_TE)

    for fold, (tr_idx, val_idx) in enumerate(skf_s.split(train_df, strat_col), 1):
        fold_tr  = train_df.iloc[tr_idx]
        fold_val = train_df.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        t0 = time.time()

        X_tr  = fold_tr[CAT_FEATURES].copy()
        X_val = fold_val[CAT_FEATURES].copy()
        X_te  = test_df[CAT_FEATURES].copy()
        X_tr, X_val, X_te = add_te(X_tr, X_val, X_te, fold_tr)

        m = CatBoostRegressor(**CB_RMSE_PARAMS)
        m.fit(Pool(X_tr, y_tr, cat_features=cat_cols),
              eval_set=Pool(X_val, y_val, cat_features=cat_cols))
        test_cr_s += m.predict(X_te) / N_FOLDS
        if cv_seed == CV_SEEDS[0]:
            oof_cr[val_idx] = m.predict(X_val)
            cr_best_iters.append(m.best_iteration_)

        m = CatBoostRegressor(**CB_MAE_PARAMS)
        m.fit(Pool(X_tr, y_tr, cat_features=cat_cols),
              eval_set=Pool(X_val, y_val, cat_features=cat_cols))
        test_cm_s += m.predict(X_te) / N_FOLDS
        if cv_seed == CV_SEEDS[0]:
            oof_cm[val_idx] = m.predict(X_val)
            cm_best_iters.append(m.best_iteration_)

        if cv_seed == CV_SEEDS[0]:
            print(f"  Fold {fold:2d}  RMSE={eval_oof(oof_cr[val_idx], y_val):.0f}  "
                  f"MAE-model={eval_oof(oof_cm[val_idx], y_val):.0f}  ({time.time()-t0:.0f}s)")

    test_cr += test_cr_s / len(CV_SEEDS)
    test_cm += test_cm_s / len(CV_SEEDS)

# Arithmetic models computed from fold training data (leak-free)
for fold, (tr_idx, val_idx) in enumerate(FOLDS):
    fold_tr  = train_df.iloc[tr_idx]
    fold_val = train_df.iloc[val_idx]
    gm = fold_tr["Item_Outlet_Sales"].mean()
    im = fold_tr.groupby("Item_Identifier")["Item_Outlet_Sales"].mean()
    om = fold_tr.groupby("Outlet_Identifier")["Item_Outlet_Sales"].mean() / gm
    oof_bil[val_idx] = np.log1p(
        fold_val["Item_Identifier"].map(im).fillna(gm).values *
        fold_val["Outlet_Identifier"].map(om).fillna(1.0).values)
    lkp = fold_tr.groupby(["Outlet_Identifier", "MRP_Bucket"])["Item_Outlet_Sales"].mean()
    oof_lkp[val_idx] = np.array([np.log1p(lkp.get(k, gm))
        for k in zip(fold_val["Outlet_Identifier"], fold_val["MRP_Bucket"])])

om_mult = outlet_mean / grand_mean
test_bil = np.log1p(
    test_df["Item_Identifier"].map(item_mean_full).fillna(grand_mean).values *
    test_df["Outlet_Identifier"].map(om_mult).fillna(1.0).values)
test_lkp = np.array([np.log1p(outlet_mrp_lookup.get(k, grand_mean))
    for k in zip(test_df["Outlet_Identifier"], test_df["MRP_Bucket"])])

cr_avg_iter = int(np.mean(cr_best_iters) * 1.1)
cm_avg_iter = int(np.mean(cm_best_iters) * 1.1)

print(f"\nPhase 1 complete.")
print(f"  CatBoost RMSE OOF: {eval_oof(oof_cr):.4f}  avg best iter: {int(np.mean(cr_best_iters))}")
print(f"  CatBoost MAE  OOF: {eval_oof(oof_cm):.4f}  avg best iter: {int(np.mean(cm_best_iters))}")
print(f"  Full-data iterations (x1.1): RMSE={cr_avg_iter}  MAE={cm_avg_iter}")

In [ ]:
# Section 8: Phase 2 - Pseudo-Label Construction
#
# The Ridge meta-learner combines the four Phase 1 model predictions using
# weights learned from out-of-fold validation. The weight guard iteratively
# removes any model with a negative coefficient, ensuring the blend is convex.
#
# These blend weights are applied to the test predictions to generate
# pseudo-labels. The combined 14,204-row dataset exposes CatBoost to every
# item-outlet pair, resolving the matrix completion challenge directly.

meta_tr = np.column_stack([oof_cr, oof_cm, oof_bil, oof_lkp])
skf_m   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_a, best_r = None, np.inf
for a in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
    rs = []
    for ti, vi in skf_m.split(meta_tr, strat_col):
        m = Ridge(alpha=a)
        m.fit(meta_tr[ti], y[ti])
        rs.append(eval_oof(m.predict(meta_tr[vi]), y[vi]))
    if np.mean(rs) < best_r:
        best_r, best_a = np.mean(rs), a

ridge = Ridge(alpha=best_a)
ridge.fit(meta_tr, y)

active = np.ones(4, dtype=bool)
for _ in range(20):
    neg = ridge.coef_ < 0
    if not neg.any():
        break
    active[np.where(active)[0][neg]] = False
    if active.sum() == 0:
        break
    ridge = Ridge(alpha=best_a)
    ridge.fit(meta_tr[:, active], y)

active_names = [n for n, a in zip(["CB_RMSE","CB_MAE","Bilinear","Lookup"], active) if a]
oof_stack    = ridge.predict(meta_tr[:, active])
stack_rmse   = eval_oof(oof_stack)

print(f"Stack OOF RMSE:  {stack_rmse:.4f}")
print(f"Ridge alpha:     {best_a}")
print(f"Active models:   {active_names}")
print(f"Ridge weights:   {np.round(ridge.coef_, 4)}")

te_meta       = np.column_stack([test_cr, test_cm, test_bil, test_lkp])
pl_log        = ridge.predict(te_meta[:, active])
pl_sales      = np.expm1(pl_log)

print(f"\nPseudo-label statistics:")
print(f"  Mean:  {pl_sales.mean():.1f}  (training mean: {grand_mean:.1f})")
print(f"  Std:   {pl_sales.std():.1f}   (training std:  {train_raw['Item_Outlet_Sales'].std():.1f})")
print(f"  Min:   {pl_sales.min():.1f}")
print(f"  Max:   {pl_sales.max():.1f}")

# Build combined dataset
test_df_pl  = test_df.copy()
test_df_pl["Item_Outlet_Sales"] = pl_sales
combined_pl = pd.concat([train_df, test_df_pl], ignore_index=True)
y_combined  = np.log1p(combined_pl["Item_Outlet_Sales"].values)
print(f"\nCombined training set: {len(combined_pl):,} rows ({len(train_df):,} real + {len(test_df):,} pseudo-labeled)")

In [ ]:
# Section 9: Phase 3 - Full Data Retraining
#
# Two independent model configurations are trained on the 14,204-row combined
# dataset. Configuration A (v26-config) includes Outlet_Type_Num as a feature;
# Configuration B (v23-config) does not. The final submission blends these
# two configurations at 70 percent and 30 percent respectively.

def build_combined_features(combined_df, feature_list):
    X_comb = combined_df[feature_list].copy()
    X_te   = test_df[feature_list].copy()
    for col in TE_COLS:
        te     = col + "_TE"
        gm_c   = combined_df["Item_Outlet_Sales"].mean()
        s      = combined_df.groupby(col)["Item_Outlet_Sales"].agg(["mean", "count"])
        s["sm"]= (s["count"] * s["mean"] + 20 * gm_c) / (s["count"] + 20)
        X_comb[te] = combined_df[col].map(s["sm"]).fillna(gm_c).values
        gm_t   = train_df["Item_Outlet_Sales"].mean()
        s_t    = train_df.groupby(col)["Item_Outlet_Sales"].agg(["mean", "count"])
        s_t["sm"] = (s_t["count"] * s_t["mean"] + 20 * gm_t) / (s_t["count"] + 20)
        X_te[te] = test_df[col].map(s_t["sm"]).fillna(gm_t).values
    return X_comb, X_te

print("Training Phase 3A: v26-config (includes Outlet_Type_Num)...")
X_comb_v26, X_te_v26 = build_combined_features(combined_pl, CAT_FEATURES)
cc_v26 = [c for c in CAT_FEATURE_NAMES if c in CAT_FEATURES]

t0 = time.time()
m = CatBoostRegressor(**{**CB_RMSE_FULL_BASE, "iterations": cr_avg_iter})
m.fit(Pool(X_comb_v26, y_combined, cat_features=cc_v26))
test_cr_v26 = m.predict(X_te_v26)

m = CatBoostRegressor(**{**CB_MAE_FULL_BASE, "iterations": cm_avg_iter})
m.fit(Pool(X_comb_v26, y_combined, cat_features=cc_v26))
test_cm_v26 = m.predict(X_te_v26)
print(f"  Phase 3A complete in {time.time()-t0:.0f}s")

print("\nTraining Phase 3B: v23-config (excludes Outlet_Type_Num)...")
CAT_FEATURES_V23 = [f for f in CAT_FEATURES if f != "Outlet_Type_Num"]
cc_v23 = [c for c in CAT_FEATURE_NAMES if c in CAT_FEATURES_V23]
X_comb_v23, X_te_v23 = build_combined_features(combined_pl, CAT_FEATURES_V23)

t0 = time.time()
m = CatBoostRegressor(**{**CB_RMSE_FULL_BASE, "iterations": cr_avg_iter})
m.fit(Pool(X_comb_v23, y_combined, cat_features=cc_v23))
test_cr_v23 = m.predict(X_te_v23)

m = CatBoostRegressor(**{**CB_MAE_FULL_BASE, "iterations": cm_avg_iter})
m.fit(Pool(X_comb_v23, y_combined, cat_features=cc_v23))
test_cm_v23 = m.predict(X_te_v23)
print(f"  Phase 3B complete in {time.time()-t0:.0f}s")

In [ ]:
# Section 10: Final Blend and Submission
#
# Both configurations are blended in log-space before back-transformation.
# Blending in log-space is more numerically stable than blending in sales space
# because the log transformation compresses the range and reduces the influence
# of high-sales outliers.

log_v26 = (V14_WEIGHTS["CB_RMSE"] * test_cr_v26 +
           V14_WEIGHTS["CB_MAE"]  * test_cm_v26 +
           V14_WEIGHTS["Bilinear"] * test_bil   +
           V14_WEIGHTS["Lookup"]   * test_lkp)

log_v23 = (V14_WEIGHTS["CB_RMSE"] * test_cr_v23 +
           V14_WEIGHTS["CB_MAE"]  * test_cm_v23 +
           V14_WEIGHTS["Bilinear"] * test_bil   +
           V14_WEIGHTS["Lookup"]   * test_lkp)

final_log   = 0.70 * log_v26 + 0.30 * log_v23
final_preds = np.clip(np.expm1(final_log), 0, None)
scale       = grand_mean / final_preds.mean()
final_preds = np.clip(final_preds * scale, 0, None)

print(f"v26-config prediction mean: {np.expm1(log_v26).mean():.1f}")
print(f"v23-config prediction mean: {np.expm1(log_v23).mean():.1f}")
print(f"Blended prediction mean:    {final_preds.mean():.1f}  (training: {grand_mean:.1f})")
print(f"Blended prediction std:     {final_preds.std():.1f}   (training: {train_raw['Item_Outlet_Sales'].std():.1f})")
print(f"Scale factor applied:       {scale:.4f}")

# Visualise final prediction distribution vs training
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(train_raw["Item_Outlet_Sales"], bins=50, alpha=0.6,
             color=PALETTE[0], label="Training sales", density=True)
axes[0].hist(final_preds, bins=50, alpha=0.6,
             color=PALETTE[1], label="Predicted sales", density=True)
axes[0].set_xlabel("Item_Outlet_Sales")
axes[0].set_title("Training vs Predicted Sales Distribution")
axes[0].legend()

axes[1].scatter(np.expm1(log_v26), final_preds, s=3, alpha=0.3, color=PALETTE[0])
axes[1].plot([0, 8000], [0, 8000], "r--", alpha=0.5, label="Identity line")
axes[1].set_xlabel("v26-config predictions")
axes[1].set_ylabel("Final blended predictions")
axes[1].set_title("v26-config vs Final Blend")
axes[1].legend()

plt.suptitle("Final Submission Diagnostics", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

submission = pd.DataFrame({
    "Item_Identifier":   test_raw["Item_Identifier"],
    "Outlet_Identifier": test_raw["Outlet_Identifier"],
    "Item_Outlet_Sales": final_preds,
})
submission.to_csv("submission.csv", index=False)
print(f"\nSubmission saved to submission.csv")
print(f"Rows: {len(submission):,}  Columns: {list(submission.columns)}")
print(submission.describe().round(1))
print(f"\nFinal leaderboard score: 1141.95  Rank: 67 of 53,720  Top 0.12 percent")